# Module 4.1: KV Caching (Optimizing Inference)

Welcome to Module 4! We've built a complete Transformer, but there's a serious problem when we try to use it for text generation (like ChatGPT): **it is incredibly slow**.

In this notebook, we will uncover the bottleneck in autoregressive generation and solve it using **Key-Value (KV) Caching**.

## 1. The Bottleneck: Autoregressive Generation

When a Decoder-only model (like GPT) generates text, it does so **autoregressively**, predicting one word at a time based on all previous words.

### The Naive Approach
1. Input: "I love"
2. Model processes "I love" -> predicts "machine"
3. Input: "I love machine"
4. Model processes "I love machine" -> predicts "learning"
5. Input: "I love machine learning"

**The Problem:** Look at step 4. To predict "learning", the model recalculates the mathematical representations (Keys and Values) for "I", "love", and "machine", even though they haven't changed since steps 2 and 3! 

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(0)

# NOTE: This `heavy_layer` is just a plain Linear standing in for a real
# Transformer block. It has NO attention, so it doesn't actually show the
# quadratic blow-up by itself — we use it only to illustrate the difference
# between *reprocessing the whole sequence* vs. *processing one new token*.
d_model = 128
heavy_layer = nn.Linear(d_model, d_model)

# Naive Generation Loop: we re-feed the ENTIRE growing sequence every step.
sequence_length = 200
tokens = torch.randn(1, 1, d_model)  # Start with 1 token

start_time = time.time()
for i in range(sequence_length):
    # We pass the ENTIRE growing sequence through the model every time.
    # The amount of work grows with the sequence length on every step.
    output = heavy_layer(tokens)

    # Take the last token's output as a stand-in "prediction", append it.
    new_token = output[:, -1:, :]
    tokens = torch.cat([tokens, new_token], dim=1)
naive_time = time.time() - start_time

print(f"Naive approach computed sequence up to length {tokens.shape[1]}")
print(f"Naive total time: {naive_time*1000:.2f} ms")

## 2. The Solution: KV Caching

In the Self-Attention mechanism (`Q * K.T * V`), the calculation for a new token only strictly needs:
1. Its own **Query (Q)** vector.
2. The **Key (K)** and **Value (V)** vectors of *all historical tokens*.

Since historical tokens do not change, their `K` and `V` vectors will always be the same. Instead of recalculating them, we can **cache** them in memory!

```mermaid
graph TD
    A[New Token: 'learning'] -->|Calculate| B(Query)
    A -->|Calculate| C(Key)
    A -->|Calculate| D(Value)
    
    C -.->|Append to Memory| E[(KV Cache Memory)]
    D -.->|Append to Memory| E
    
    B -->|Attention Dot Product| E
```

In [ ]:
class AttentionWithKVCache(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.d_model = d_model
        
    def forward(self, x, kv_cache=None):
        """
        x: The NEW token(s) being passed in. Shape: (Batch, Seq_Len, d_model)
           During generation, Seq_Len is always 1!
        kv_cache: A tuple of (cached_K, cached_V) from previous steps.
        """
        # Calculate Q, K, V for the currently inputted token(s) ONLY
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        if kv_cache is not None:
            # Unpack the cache
            past_K, past_V = kv_cache
            
            # CONCATENATE the newly calculated K, V to the historical cache
            K = torch.cat([past_K, K], dim=1) # Concat along sequence dimension
            V = torch.cat([past_V, V], dim=1)
            
        # Update our cache for the NEXT token generation step
        new_kv_cache = (K, V)
        
        # Perform Standard Attention.
        # Notice: NO causal mask here! During single-token decoding the new
        # token's Query is *supposed* to attend to ALL past Keys (everything in
        # the cache is in the past). That's exactly why caching is so clean here.
        # The causal mask only matters during "prefill" — when we process the
        # whole prompt at once and a token must not peek at tokens after it.
        # Q is shape (Batch, 1, d_model)
        # K is shape (Batch, Total_History_Len, d_model)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_model ** 0.5)
        attention_weights = torch.softmax(scores, dim=-1)
        
        # V is shape (Batch, Total_History_Len, d_model)
        output = torch.matmul(attention_weights, V)
        
        return output, new_kv_cache

print("KV Cache Attention mechanism initialized!")

## 3. Simulating Generation with KV Cache

Let's see how the loop changes when we use a KV Cache. Notice how we only ever pass a vector of length `1` into the model at each step!

In [ ]:
model = AttentionWithKVCache(d_model=d_model)  # reuse the same d_model = 128
batch_size = 1

# Step 1: Initialize the very first token to start generation
current_token = torch.randn(batch_size, 1, d_model)
kv_cache = None

print("Starting Autoregressive Generation...")
for i in range(5):
    # We ONLY pass in the current_token (sequence length is always 1!)
    # We also pass the memory of the past.
    output, kv_cache = model(current_token, kv_cache)
    
    # The Cache grows in the background! 
    # past_K is at index 0 of the tuple. Its shape is (Batch, Seq_Len, d_model)
    current_cache_size = kv_cache[0].shape[1]
    print(f"Step {i+1}: Generated 1 token. Current Cache Sequence Length: {current_cache_size}")
    
    # IMPORTANT — this is a simplification:
    # In a REAL generation loop, we would (1) project `output` to vocab-sized
    # logits, (2) sample/argmax to pick the next token ID, then (3) look that ID
    # up in the embedding table to get the next input vector.
    # Here we skip all of that and just feed `output` straight back in. We do
    # this only to GROW THE CACHE so you can watch its sequence length climb.
    current_token = output 

## Summary

By implementing **KV Caching**, we changed the cost of generating each new token. Instead of reprocessing the whole sequence every step, we reuse the Keys and Values we already computed and only do fresh work for the single new token.

One thing to keep in mind: the cache we built here stored one K and one V per token. A *real* model has many attention heads, so the cache is multiplied by `num_heads` — every head stores its own Keys and Values. That makes the cache much larger, which is exactly the problem the next notebook tackles.

Saving all this data to memory creates a new challenge: **we need a lot of VRAM to store the KV Cache for long conversations or many concurrent users!**

This sets us up for **Module 4.2: Advanced Attention (GQA and MQA)**, where we learn how to shrink the size of the KV Cache to save GPU memory.

### 🏋️ Try it yourself

The cache reuses past Keys and Values instead of recomputing them. Prove to yourself that the cached path produces the **same** result as recomputing everything from scratch.

Starter idea:
1. Run the cached loop for a few steps and keep every `current_token` you fed in.
2. Stack those tokens into one tensor and run them through `W_q`, `W_k`, `W_v` *all at once* (no cache), then do attention for the last position only.
3. Compare that last-position output to the cached `output` of the final step with `torch.allclose(...)`. They should match (up to tiny floating-point error).

In [ ]:
# Your turn: confirm the cache gives the same answer as recomputing from scratch.
torch.manual_seed(0)
verify_model = AttentionWithKVCache(d_model=d_model)

# 1) Run the cached loop, remembering every input token we fed in.
fed_tokens = []
tok = torch.randn(1, 1, d_model)
cache = None
for _ in range(5):
    fed_tokens.append(tok)
    cached_out, cache = verify_model(tok, cache)
    tok = cached_out

# 2) Recompute everything in one shot (no cache) for the full sequence.
full_seq = torch.cat(fed_tokens, dim=1)          # (1, 5, d_model)
Q = verify_model.W_q(full_seq)
K = verify_model.W_k(full_seq)
V = verify_model.W_v(full_seq)
scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_model ** 0.5)
weights = torch.softmax(scores, dim=-1)
full_out = torch.matmul(weights, V)

# 3) The cached output of the LAST step should match the last position here.
print("Match:", torch.allclose(cached_out, full_out[:, -1:, :], atol=1e-5))